# Supervised Hierarchical Rao-Style PC-CNN — V2

## Free inference + short supervised nudging + independent local classifier learning

This notebook matches `pc_supervised_rao_final_v2.py`.

Key design:

- `U1` and `U2` use Rao-style local predictive learning and Rao `k2`.
- Classification uses `W3 z + b3`.
- `W3` and `b3` use their own local supervised learning rate.
- `W3/b3` are trained from **free `r2`**, the same representation used at validation/test time.
- Classification error still nudges `r2` for a short supervised phase so supervision can influence the predictive-coding backbone.
- No autograd, no backpropagation, no optimizer.


In [ ]:
##%%
import copy
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from collections import defaultdict
from torch.utils.data import DataLoader, Subset

import src.data_cleaning_and_preprocessing._model_ready_dataset as model_ready_dataset
from src.dataset.tiny_imagenet_dataset import TinyImageNetLoader
from archive.py_file.pc_supervised_rao_6 import HierarchicalPCCNN

In [ ]:
##%%
SEED = 42
BATCH_SIZE = 16
IMAGE_SIZE = 64

NUM_CLASSES = 20
TRAIN_IMAGES_PER_CLASS = 200
VAL_IMAGES_PER_CLASS = 50
TEST_IMAGES_PER_CLASS = 50

INFERENCE_STEPS = 100
SUPERVISED_NUDGE_STEPS = 50
PC_EPOCHS = 15

SIGMA_CLASS_SQ = 1.0
CLASS_POOL_SIZE = 4
CLASS_LEARNING_RATE = 0.01

In [ ]:
##%%
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available() and hasattr(torch.mps, "manual_seed"):
    torch.mps.manual_seed(SEED)

In [ ]:
##%%
cleaned_tiny_imagenet = model_ready_dataset.ModelReadyDatasetLoader(TinyImageNetLoader(seed=SEED), image_size=IMAGE_SIZE).load()

train_dataset = cleaned_tiny_imagenet[0]
test_dataset = cleaned_tiny_imagenet[1]

print("Full training dataset:", len(train_dataset))
print("Full test dataset:", len(test_dataset))

In [ ]:
##%%
selected_classes = list(range(NUM_CLASSES))

print("Selected classes:", selected_classes)
print("Number of selected classes:", len(selected_classes))

In [ ]:
##%%
class_indices = defaultdict(list)

for i in range(len(train_dataset)):
    _, label = train_dataset[i]
    label = int(label)

    if label in selected_classes:
        class_indices[label].append(i)

In [ ]:
##%%
split_generator = torch.Generator()
split_generator.manual_seed(SEED)

train_indices = []
val_indices = []

for class_id in selected_classes:
    available_indices = class_indices[class_id]
    required_images = TRAIN_IMAGES_PER_CLASS + VAL_IMAGES_PER_CLASS

    if len(available_indices) < required_images:
        raise ValueError(f"Class {class_id} has only {len(available_indices)} images, but {required_images} are required.")

    permutation = torch.randperm(len(available_indices), generator=split_generator)

    train_selection = permutation[:TRAIN_IMAGES_PER_CLASS]
    val_selection = permutation[TRAIN_IMAGES_PER_CLASS:TRAIN_IMAGES_PER_CLASS + VAL_IMAGES_PER_CLASS]

    train_selected = [available_indices[i] for i in train_selection.tolist()]
    val_selected = [available_indices[i] for i in val_selection.tolist()]

    train_indices.extend(train_selected)
    val_indices.extend(val_selected)

In [ ]:
##%%
train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(train_dataset, val_indices)

overlap = set(train_indices).intersection(set(val_indices))

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Train/Validation overlap:", len(overlap))

assert len(overlap) == 0

In [ ]:
##%%
train_loader_generator = torch.Generator()
train_loader_generator.manual_seed(SEED)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, generator=train_loader_generator)
train_eval_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

print("Training batches:", len(train_loader))
print("Training evaluation batches:", len(train_eval_loader))
print("Validation batches:", len(val_loader))

In [ ]:
##%%
test_class_indices = defaultdict(list)

for i in range(len(test_dataset)):
    _, label = test_dataset[i]
    label = int(label)

    if label in selected_classes:
        test_class_indices[label].append(i)

test_generator = torch.Generator()
test_generator.manual_seed(SEED)

test_indices = []

for class_id in selected_classes:
    available_indices = test_class_indices[class_id]

    if len(available_indices) < TEST_IMAGES_PER_CLASS:
        raise ValueError(f"Test class {class_id} has only {len(available_indices)} images, but {TEST_IMAGES_PER_CLASS} were requested.")

    permutation = torch.randperm(len(available_indices), generator=test_generator)
    selection = permutation[:TEST_IMAGES_PER_CLASS]

    selected = [available_indices[i] for i in selection.tolist()]
    test_indices.extend(selected)

test_subset = Subset(test_dataset, test_indices)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False)

print("Final test images:", len(test_subset))

In [ ]:
##%%
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available() and hasattr(torch.mps, "manual_seed"):
    torch.mps.manual_seed(SEED)

model = HierarchicalPCCNN(num_classes=NUM_CLASSES, input_size=IMAGE_SIZE, inference_steps=INFERENCE_STEPS, supervised_nudge_steps=SUPERVISED_NUDGE_STEPS, convergence_tolerance=1e-5, sigma_class_sq=SIGMA_CLASS_SQ, class_pool_size=CLASS_POOL_SIZE, class_learning_rate=CLASS_LEARNING_RATE).to(device)

print(model)
print("sigma_bu_sq:", model.sigma_bu_sq)
print("sigma_td_sq:", model.sigma_td_sq)
print("sigma_class_sq:", model.sigma_class_sq)
print("class_learning_rate:", model.class_learning_rate)
print("free inference steps:", model.inference_steps)
print("supervised nudge steps:", model.supervised_nudge_steps)

In [ ]:
##%%
total_params = sum(parameter.numel() for parameter in model.parameters())
autograd_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

print("Total unique parameters:", total_params)
print("Autograd-enabled parameters:", autograd_params)

assert autograd_params == 0

In [ ]:
##%%
initial_u1 = model.level1["conv"].weight.detach().cpu().clone()
initial_u2 = model.level2["conv"].weight.detach().cpu().clone()
initial_u3 = model.level3["linear"].weight.detach().cpu().clone()
initial_b3 = model.level3["linear"].bias.detach().cpu().clone()

print("Initial U1 norm:", initial_u1.norm().item())
print("Initial U2 norm:", initial_u2.norm().item())
print("Initial U3 norm:", initial_u3.norm().item())
print("Initial b3:", initial_b3)
print("Initial training_input_count:", model.training_input_count)
print("Initial k2:", model.k2)

In [ ]:
##%%
@torch.no_grad()
def evaluate_model(model, data_loader, device, name=""):
    total_loss = 0.0
    total_correct = 0
    total_images = 0
    total_steps = 0
    total_converged = 0

    model.eval()

    for images, labels in tqdm(data_loader, desc=f"Evaluation Supervised Rao PC-CNN - {name}"):
        images = images.to(device)
        labels = labels.to(device=device, dtype=torch.long)

        result = model.classify_batch(images)

        probabilities = result["probabilities"]
        predictions = result["predictions"]

        true_probabilities = probabilities[torch.arange(labels.shape[0], device=labels.device), labels]
        loss = -torch.log(true_probabilities + 1e-8).mean().item()

        total_loss += loss * labels.shape[0]
        total_correct += (predictions == labels).sum().item()
        total_images += labels.shape[0]

        total_steps += sum(result["inference_steps"])
        total_converged += sum(int(value) for value in result["converged"])

    return total_loss / total_images, 100.0 * total_correct / total_images, total_steps / total_images, 100.0 * total_converged / total_images

In [ ]:
##%%
history = []

best_val_accuracy = -1.0
best_val_loss = float("inf")

best_epoch = None
best_model_state = None
best_k2 = None
best_training_input_count = None

for epoch in range(PC_EPOCHS):
    total_error_0 = 0.0
    total_error_1 = 0.0
    total_error_2 = 0.0

    total_nudged_error_0 = 0.0
    total_nudged_error_1 = 0.0

    total_class_loss = 0.0

    total_free_steps = 0
    total_nudge_steps = 0
    total_converged = 0
    total_images = 0

    model.train()

    progress_bar = tqdm(train_loader, desc=f"Supervised Rao PC-CNN Epoch {epoch + 1}/{PC_EPOCHS}")

    for images, labels in progress_bar:
        for image, label in zip(images, labels):
            image = image.to(device)
            label = label.to(device=device, dtype=torch.long)

            result = model(image, label)

            total_error_0 += result["error_0"]
            total_error_1 += result["error_1"]
            total_error_2 += result["error_2"]

            total_nudged_error_0 += result["nudged_error_0"]
            total_nudged_error_1 += result["nudged_error_1"]

            total_class_loss += result["class_loss"]

            total_free_steps += result["inference_steps"]
            total_nudge_steps += result["nudge_steps"]
            total_converged += int(result["converged"])
            total_images += 1

    mean_error_0 = total_error_0 / total_images
    mean_error_1 = total_error_1 / total_images
    mean_error_2 = total_error_2 / total_images

    mean_nudged_error_0 = total_nudged_error_0 / total_images
    mean_nudged_error_1 = total_nudged_error_1 / total_images

    mean_class_loss = total_class_loss / total_images

    mean_free_steps = total_free_steps / total_images
    mean_nudge_steps = total_nudge_steps / total_images
    training_convergence_rate = 100.0 * total_converged / total_images

    train_loss, train_accuracy, train_eval_steps, train_eval_convergence = evaluate_model(model, train_eval_loader, device, "Train")
    val_loss, val_accuracy, val_steps, val_convergence = evaluate_model(model, val_loader, device, "Validation")

    history.append({"epoch": epoch + 1, "error_0": mean_error_0, "error_1": mean_error_1, "error_2": mean_error_2, "nudged_error_0": mean_nudged_error_0, "nudged_error_1": mean_nudged_error_1, "class_loss": mean_class_loss, "mean_free_steps": mean_free_steps, "mean_nudge_steps": mean_nudge_steps, "training_convergence_rate": training_convergence_rate, "train_loss": train_loss, "train_accuracy": train_accuracy, "train_eval_steps": train_eval_steps, "train_eval_convergence": train_eval_convergence, "val_loss": val_loss, "val_accuracy": val_accuracy, "val_steps": val_steps, "val_convergence": val_convergence, "k2": model.k2})

    # is_better = val_accuracy > best_val_accuracy
    #
    # if val_accuracy == best_val_accuracy and val_loss < best_val_loss:
    #     is_better = True

    is_better = val_loss < best_val_loss

    if val_loss == best_val_loss and val_accuracy > best_val_accuracy:
        is_better = True

    if is_better:
        best_val_accuracy = val_accuracy
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_model_state = copy.deepcopy(model.state_dict())
        best_k2 = model.k2
        best_training_input_count = model.training_input_count

    print(f"Epoch {epoch + 1}/{PC_EPOCHS} | e0: {mean_error_0:.6f} | e1: {mean_error_1:.6f} | e2: {mean_error_2:.6f} | Nudge e0: {mean_nudged_error_0:.6f} | Nudge e1: {mean_nudged_error_1:.6f} | Class Loss: {mean_class_loss:.4f} | Train Acc: {train_accuracy:.2f}% | Val Acc: {val_accuracy:.2f}% | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | k2: {model.k2:.8f}")

In [ ]:
##%%
trained_u1 = model.level1["conv"].weight.detach().cpu()
trained_u2 = model.level2["conv"].weight.detach().cpu()
trained_u3 = model.level3["linear"].weight.detach().cpu()
trained_b3 = model.level3["linear"].bias.detach().cpu()

print("AFTER TRAINING")
print("training_input_count:", model.training_input_count)
print("k2:", model.k2)

print("U1 norm:", trained_u1.norm().item())
print("U2 norm:", trained_u2.norm().item())
print("U3 norm:", trained_u3.norm().item())
print("b3:", trained_b3)

print("U1 change:", torch.norm(trained_u1 - initial_u1).item())
print("U2 change:", torch.norm(trained_u2 - initial_u2).item())
print("U3 change:", torch.norm(trained_u3 - initial_u3).item())
print("b3 change:", torch.norm(trained_b3 - initial_b3).item())

In [ ]:
##%%
if best_model_state is None:
    raise RuntimeError("No model checkpoint was saved.")

model.load_state_dict(best_model_state)
model.k2 = best_k2
model.training_input_count = best_training_input_count
model.eval()

print("Best epoch:", best_epoch)
print("Best validation accuracy:", f"{best_val_accuracy:.2f}%")
print("Best validation loss:", f"{best_val_loss:.4f}")

In [ ]:
##%%
print("=" * 60)
print("BEST CHECKPOINT DIAGNOSTICS")
print("=" * 60)

print("Best epoch:", best_epoch)
print("training_input_count:", model.training_input_count)
print("k2:", model.k2)

print("U1 norm:", model.level1["conv"].weight.norm().item())
print("U2 norm:", model.level2["conv"].weight.norm().item())
print("U3 norm:", model.level3["linear"].weight.norm().item())
print("b3:", model.level3["linear"].bias.detach().cpu())

print("U1 change from initialization:", torch.norm(model.level1["conv"].weight.detach().cpu() - initial_u1).item())
print("U2 change from initialization:", torch.norm(model.level2["conv"].weight.detach().cpu() - initial_u2).item())
print("U3 change from initialization:", torch.norm(model.level3["linear"].weight.detach().cpu() - initial_u3).item())
print("b3 change from initialization:", torch.norm(model.level3["linear"].bias.detach().cpu() - initial_b3).item())

In [ ]:
##%%
@torch.no_grad()
def evaluate_per_class(model, data_loader, device, num_classes):
    class_correct = torch.zeros(num_classes, dtype=torch.long)
    class_total = torch.zeros(num_classes, dtype=torch.long)

    model.eval()

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device=device, dtype=torch.long)

        result = model.classify_batch(images)
        predictions = result["predictions"]

        for class_id in range(num_classes):
            mask = labels == class_id
            class_correct[class_id] += (predictions[mask] == labels[mask]).sum().cpu()
            class_total[class_id] += mask.sum().cpu()

    for class_id in range(num_classes):
        accuracy = 100.0 * class_correct[class_id].item() / class_total[class_id].item()
        print(f"Class {class_id} | Correct: {class_correct[class_id].item()}/{class_total[class_id].item()} | Accuracy: {accuracy:.2f}%")

evaluate_per_class(model, val_loader, device, NUM_CLASSES)

In [ ]:
##%%
with torch.no_grad():
    for class_id in range(NUM_CLASSES):
        print(f"U3 class {class_id} norm:", model.level3["linear"].weight[class_id].norm().item())

with torch.no_grad():
    u3 = model.level3["linear"].weight.detach()
    similarity = F.cosine_similarity(u3.unsqueeze(1), u3.unsqueeze(0), dim=2)

print("U3 class-weight cosine similarity:")
print(similarity.cpu())

In [ ]:
##%%
model.eval()

for class_id in selected_classes:
    for image, label in train_subset:
        if int(label) == class_id:
            result = model.classify(image.to(device))

            print("True class:", class_id)
            print("Scores:", result["scores"].detach().cpu().flatten())
            print("Probabilities:", result["probabilities"].detach().cpu().flatten())
            print("Predicted class:", result["prediction"].item())
            print("Free inference steps:", result["inference_steps"])
            print("Converged:", result["converged"])
            print()
            break

In [ ]:
@torch.no_grad()
def create_confusion_matrix(model, data_loader, device, num_classes):
    confusion_matrix = torch.zeros((num_classes, num_classes), dtype=torch.long)

    model.eval()

    for images, labels in tqdm(data_loader, desc="Validation Confusion Matrix"):
        images = images.to(device)
        labels = labels.to(device=device, dtype=torch.long)

        result = model.classify_batch(images)
        predictions = result["predictions"]

        for true_label, predicted_label in zip(labels, predictions):
            confusion_matrix[true_label.item(), predicted_label.item()] += 1

    return confusion_matrix


val_confusion_matrix = create_confusion_matrix(model, val_loader, device, NUM_CLASSES)

print("Validation Confusion Matrix:")
print(val_confusion_matrix)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

image = ax.imshow(val_confusion_matrix.numpy())

ax.set_title("Validation Confusion Matrix")
ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))

ax.set_xticklabels([f"Class {i}" for i in range(NUM_CLASSES)])
ax.set_yticklabels([f"Class {i}" for i in range(NUM_CLASSES)])

for true_class in range(NUM_CLASSES):
    for predicted_class in range(NUM_CLASSES):
        value = val_confusion_matrix[true_class, predicted_class].item()
        ax.text(predicted_class, true_class, str(value), ha="center", va="center")

plt.tight_layout()
plt.show()

In [ ]:
saved_bias = model.level3["linear"].bias.detach().clone()

print("CURRENT BIAS")
print(saved_bias.cpu())

original_val_loss, original_val_accuracy, _, _ = evaluate_model(model, val_loader, device, "Original Bias")
original_confusion_matrix = create_confusion_matrix(model, val_loader, device, NUM_CLASSES)

with torch.no_grad():
    model.level3["linear"].bias.zero_()

zero_bias_val_loss, zero_bias_val_accuracy, _, _ = evaluate_model(model, val_loader, device, "Zero Bias")
zero_bias_confusion_matrix = create_confusion_matrix(model, val_loader, device, NUM_CLASSES)

print()
print("ORIGINAL BIAS")
print("Validation Loss:", f"{original_val_loss:.4f}")
print("Validation Accuracy:", f"{original_val_accuracy:.2f}%")
print(original_confusion_matrix)

print()
print("ZERO BIAS")
print("Validation Loss:", f"{zero_bias_val_loss:.4f}")
print("Validation Accuracy:", f"{zero_bias_val_accuracy:.2f}%")
print(zero_bias_confusion_matrix)

with torch.no_grad():
    model.level3["linear"].bias.copy_(saved_bias)

In [ ]:
print("True counts:", val_confusion_matrix.sum(dim=1))
print("Predicted counts:", val_confusion_matrix.sum(dim=0))

In [ ]:
with torch.no_grad():
    model.level3["linear"].bias.copy_(saved_bias)

In [ ]:
@torch.no_grad()
def collect_class_features(model, data_loader, device, num_classes):
    features_by_class = [[] for _ in range(num_classes)]

    model.eval()

    for images, labels in tqdm(data_loader, desc="Collecting Free r2 Features"):
        for image, label in zip(images, labels):
            image = image.to(device)
            label = int(label)

            _, r2, _, _ = model.infer(image)
            features = model.class_features(r2).squeeze(0).detach().cpu()

            features_by_class[label].append(features)

    features_by_class = [torch.stack(features) for features in features_by_class]

    return features_by_class


val_features_by_class = collect_class_features(model, val_loader, device, NUM_CLASSES)

In [ ]:
class_centroids = torch.stack([features.mean(dim=0) for features in val_features_by_class])

normalized_centroids = F.normalize(class_centroids, p=2, dim=1)
centroid_similarity = normalized_centroids @ normalized_centroids.T

print("Free r2 Class-Centroid Cosine Similarity:")
print(centroid_similarity)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

image = ax.imshow(centroid_similarity.numpy(), vmin=-1, vmax=1)

ax.set_title("Free r2 Class-Centroid Cosine Similarity")
ax.set_xlabel("Class")
ax.set_ylabel("Class")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))

ax.set_xticklabels([f"Class {i}" for i in range(NUM_CLASSES)])
ax.set_yticklabels([f"Class {i}" for i in range(NUM_CLASSES)])

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{centroid_similarity[i, j]:.2f}", ha="center", va="center")

plt.colorbar(image)
plt.tight_layout()
plt.show()

In [ ]:
for class_id in range(NUM_CLASSES):
    features = val_features_by_class[class_id]
    centroid = class_centroids[class_id]

    distances = torch.norm(features - centroid, dim=1)

    print(
        f"Class {class_id} | Mean distance to centroid: {distances.mean().item():.4f} | Std: {distances.std().item():.4f}")

In [ ]:
global_centroid = torch.cat(val_features_by_class, dim=0).mean(dim=0)

centered_centroids = class_centroids - global_centroid

normalized_centered_centroids = F.normalize(centered_centroids, p=2, dim=1)
centered_centroid_similarity = normalized_centered_centroids @ normalized_centered_centroids.T

print("Centered Free r2 Class-Centroid Cosine Similarity:")
print(centered_centroid_similarity)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

image = ax.imshow(centered_centroid_similarity.numpy(), vmin=-1, vmax=1)

ax.set_title("Centered Free r2 Class-Centroid Cosine Similarity")
ax.set_xlabel("Class")
ax.set_ylabel("Class")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))

ax.set_xticklabels([f"Class {i}" for i in range(NUM_CLASSES)])
ax.set_yticklabels([f"Class {i}" for i in range(NUM_CLASSES)])

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{centered_centroid_similarity[i, j]:.2f}", ha="center", va="center")

plt.colorbar(image)
plt.tight_layout()
plt.show()

In [ ]:
centroid_distances = torch.cdist(class_centroids, class_centroids, p=2)

print("Free r2 Class-Centroid Euclidean Distance:")
print(centroid_distances)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

image = ax.imshow(centroid_distances.numpy())

ax.set_title("Free r2 Class-Centroid Euclidean Distance")
ax.set_xlabel("Class")
ax.set_ylabel("Class")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))

ax.set_xticklabels([f"Class {i}" for i in range(NUM_CLASSES)])
ax.set_yticklabels([f"Class {i}" for i in range(NUM_CLASSES)])

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{centroid_distances[i, j]:.2f}", ha="center", va="center")

plt.colorbar(image)
plt.tight_layout()
plt.show()

In [ ]:
##%%
final_train_loss, final_train_accuracy, final_train_steps, final_train_convergence = evaluate_model(model, train_eval_loader, device, "Train")
final_val_loss, final_val_accuracy, final_val_steps, final_val_convergence = evaluate_model(model, val_loader, device, "Validation")

print("RESTORED BEST MODEL")
print("Train Loss:", f"{final_train_loss:.4f}")
print("Train Accuracy:", f"{final_train_accuracy:.2f}%")
print("Validation Loss:", f"{final_val_loss:.4f}")
print("Validation Accuracy:", f"{final_val_accuracy:.2f}%")

In [ ]:
##%%
test_loss, test_accuracy, test_steps, test_convergence = evaluate_model(model, test_loader, device, "Test")

print("FINAL CLEAN TEST RESULT")
print("Test Loss:", f"{test_loss:.4f}")
print("Test Accuracy:", f"{test_accuracy:.2f}%")
print("Mean Free Inference Steps:", f"{test_steps:.2f}")
print("Convergence Rate:", f"{test_convergence:.2f}%")

In [ ]:
##%%
def to_display_image(image):
    image = image.detach().cpu().clamp(0, 1)
    return image.permute(1, 2, 0).numpy()

model.eval()

class_examples = {}

for i in range(len(train_subset)):
    image, label = train_subset[i]
    label = int(label)

    if label not in class_examples:
        class_examples[label] = image

    if len(class_examples) == NUM_CLASSES:
        break

fig, axes = plt.subplots(NUM_CLASSES, 2, figsize=(10, NUM_CLASSES * 4))

for row, class_id in enumerate(selected_classes):
    image = class_examples[class_id]

    reconstruction, predicted_r1, r2, probabilities, predicted_class, steps, converged = model.reconstruct(image)

    axes[row, 0].imshow(to_display_image(image))
    axes[row, 0].set_title(f"Original | True Class {class_id}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(to_display_image(reconstruction))
    axes[row, 1].set_title(f"Reconstruction | Predicted Class {predicted_class}")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
##%%
epochs_plot = [item["epoch"] for item in history]
train_accuracy_plot = [item["train_accuracy"] for item in history]
val_accuracy_plot = [item["val_accuracy"] for item in history]

plt.figure(figsize=(8, 5))
plt.plot(epochs_plot, train_accuracy_plot, marker="o", label="Train Accuracy")
plt.plot(epochs_plot, val_accuracy_plot, marker="o", label="Validation Accuracy")
plt.axhline(100.0 / NUM_CLASSES, linestyle="--", label="Chance Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Supervised Rao PC-CNN Classification Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
##%%
train_loss_plot = [item["train_loss"] for item in history]
val_loss_plot = [item["val_loss"] for item in history]
class_loss_plot = [item["class_loss"] for item in history]

plt.figure(figsize=(8, 5))
plt.plot(epochs_plot, train_loss_plot, marker="o", label="Train Evaluation Loss")
plt.plot(epochs_plot, val_loss_plot, marker="o", label="Validation Loss")
plt.plot(epochs_plot, class_loss_plot, marker="o", label="Online Free-State Class Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Supervised Rao PC-CNN Classification Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
##%%
error_0_plot = [item["error_0"] for item in history]
error_1_plot = [item["error_1"] for item in history]
error_2_plot = [item["error_2"] for item in history]

plt.figure(figsize=(8, 5))
plt.plot(epochs_plot, error_0_plot, marker="o", label="e0: Free Image Error")
plt.plot(epochs_plot, error_1_plot, marker="o", label="e1: Free r1 Error")
plt.plot(epochs_plot, error_2_plot, marker="o", label="e2: Free Class Error")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("Free-State Prediction Errors")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
##%%
nudged_error_0_plot = [item["nudged_error_0"] for item in history]
nudged_error_1_plot = [item["nudged_error_1"] for item in history]

plt.figure(figsize=(8, 5))
plt.plot(epochs_plot, nudged_error_0_plot, marker="o", label="Nudged e0")
plt.plot(epochs_plot, nudged_error_1_plot, marker="o", label="Nudged e1")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("Supervised-Nudged Predictive Errors")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
##%%
k2_plot = [item["k2"] for item in history]

plt.figure(figsize=(8, 5))
plt.plot(epochs_plot, k2_plot, marker="o")
plt.xlabel("Epoch")
plt.ylabel("k2")
plt.title("Rao k2 Learning-Rate Schedule for U1/U2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
##%%
print("=" * 72)
print("SUPERVISED HIERARCHICAL RAO PC-CNN V2 EXPERIMENT SUMMARY")
print("=" * 72)

print("Device:", device)
print("Input size:", IMAGE_SIZE)
print("Number of classes:", NUM_CLASSES)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Test images:", len(test_subset))

print("Training epochs:", PC_EPOCHS)
print("Free inference steps:", INFERENCE_STEPS)
print("Supervised nudge steps:", SUPERVISED_NUDGE_STEPS)
print("Convergence tolerance:", model.convergence_tolerance)

print("sigma_class_sq:", model.sigma_class_sq)
print("class_learning_rate:", model.class_learning_rate)

print("Best epoch:", best_epoch)
print("Best validation accuracy:", f"{best_val_accuracy:.2f}%")
print("Best validation loss:", f"{best_val_loss:.4f}")

print("Final training accuracy:", f"{final_train_accuracy:.2f}%")
print("Final validation accuracy:", f"{final_val_accuracy:.2f}%")

print("Final clean test loss:", f"{test_loss:.4f}")
print("Final clean test accuracy:", f"{test_accuracy:.2f}%")
print("Final clean test convergence:", f"{test_convergence:.2f}%")

print("Training presentations at best epoch:", model.training_input_count)
print("k2 at best epoch:", model.k2)

print("=" * 72)